In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import logging
import os

# Add project root to path to import local modules
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.data_loader import load_data, save_data
from src.utils.helpers import setup_logging
from src.utils.config import TARGET_COLUMN, TIME_COLUMN, PROCESSED_DATA_PATH, RAW_DATA_PATH, VISUALIZATIONS_DIR

sns.set_style("whitegrid")

# Setup logging
setup_logging()
logger = logging.getLogger(__name__)

logger.info("Starting EDA Notebook execution.")

2025-11-06 14:41:43,186 - root - INFO - Logging setup complete.
2025-11-06 14:41:43,189 - __main__ - INFO - Starting EDA Notebook execution.


## 01 - Data Exploration and Initial Understanding

This notebook is dedicated to thoroughly understanding the raw dynamic pricing dataset provided. We will load the data, inspect its basic properties, identify missing values, check for anomalies, and visualize key distributions. This step is crucial for informing subsequent data preprocessing and feature engineering decisions

In [5]:
# Load the raw data

# Data Loading Loop for Each Town 
logger.info("Starting raw data loading for all towns...")

try:
    df = load_data(file_path=RAW_DATA_PATH)
    logger.info(f"Data successfully loaded for EDA. Initial Shape: {df.shape}")
except FileNotFoundError as e:
    logger.error(f"Notebook ERROR: {e}")
    logger.error("Please ensure 'rides_raw.csv' is in 'data/raw/' directory and try again.")

2025-11-06 14:41:45,401 - __main__ - INFO - Starting raw data loading for all towns...
2025-11-06 14:41:45,405 - src.data.data_loader - INFO - Loading data from: c:\Users\BRAIN\Desktop\dynamic_pricing_project\data\raw\dynamic_pricing.csv
2025-11-06 14:41:45,414 - src.data.data_loader - INFO - Data loaded successfully. Shape: (1000, 10)
2025-11-06 14:41:45,416 - __main__ - INFO - Data successfully loaded for EDA. Initial Shape: (1000, 10)


In [6]:
import io

logger.info("Initial Data Checks")

# Capture DataFrame info in a string buffer
buffer = io.StringIO()
df.info(buf=buffer)
info_str = buffer.getvalue()

logger.info("\nDataFrame Info:\n" + "\n".join(buffer))
logger.info("\nFirst 5 rows:\n" + str(df.head().to_markdown()))

# Descriptive statistics for numerical columns
logger.info("\n Descriptive Statistics for Numerical Features:\n" + str(df.describe().T.to_markdown()))

# Unique values per column
logger.info("\n Unique Values Count per Column:\n" + str(df.nunique().to_markdown()))

# Check for missing values
missing_summary = df.isnull().sum()
missing_percentage = (df.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "Missing Count": missing_summary,
    "Missing %": missing_percentage
})
logger.info("\n Missing Values Summary:\n" + str(missing_report.to_markdown()))

# Check for duplicate rows
duplicate_count = df.duplicated().sum()
logger.info(f"\n Duplicate Rows Found: {duplicate_count}")

if duplicate_count > 0:
    logger.warning("Duplicate rows detected! Consider reviewing data cleaning steps.")
else:
    logger.info(" No duplicate rows detected.")

logger.info(" Data quality checks completed successfully.")


2025-11-06 14:41:46,596 - __main__ - INFO - Initial Data Checks
2025-11-06 14:41:46,604 - __main__ - INFO - 
DataFrame Info:

2025-11-06 14:41:46,611 - __main__ - INFO - 
First 5 rows:
|    |   Number_of_Riders |   Number_of_Drivers | Location_Category   | Customer_Loyalty_Status   |   Number_of_Past_Rides |   Average_Ratings | Time_of_Booking   | Vehicle_Type   |   Expected_Ride_Duration |   Historical_Cost_of_Ride |
|---:|-------------------:|--------------------:|:--------------------|:--------------------------|-----------------------:|------------------:|:------------------|:---------------|-------------------------:|--------------------------:|
|  0 |                 90 |                  45 | Urban               | Silver                    |                     13 |              4.47 | Night             | Premium        |                       90 |                   284.257 |
|  1 |                 58 |                  39 | Suburban            | Silver                    |     

### Univariate Analysis (Numerical Features)

In this step, we perform Univariate Analysis on all numerical features of the dataset.
The goal is to understand each numeric variable independently. its distribution, spread, and potential outliers.

This helps us:

Detect skewness (whether data is normally distributed or not)

Identify outliers that could affect model training

Understand value concentration (e.g., high peaks may indicate categorical-like numeric features)

In [7]:
logger.info("Univariate Analysis: Numerical Distributions")

# Define save path for the plot
exploration_dir = os.path.join(VISUALIZATIONS_DIR, "exploration")
save_path = os.path.join(exploration_dir, "univariate_analysis_plot.png")

# Identify numeric columns (excluding target)
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if TARGET_COLUMN in numerical_cols:
    numerical_cols.remove(TARGET_COLUMN)

if not numerical_cols:
    logger.warning("No numerical columns found for univariate analysis.")
else:
    num_plots = len(numerical_cols)
    n_cols = 2
    n_rows = int(np.ceil(num_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = np.array(axes).flatten()  # Makes sure axes are iterable even for single row

    for i, col in enumerate(numerical_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], bins=30, color="#007acc", edgecolor="black")
        axes[i].set_title(f"Distribution of {col}", fontsize=12, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Frequency")

        # Add skewness annotation
        skewness = df[col].skew()
        axes[i].text(0.95, 0.95, f"Skew: {skewness:.2f}", ha='right', va='top', 
                     transform=axes[i].transAxes, fontsize=10)

    # Remove unused plots
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    # --- Save the figure ---
    os.makedirs(exploration_dir, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    logger.info(f"Univariate analysis completed and saved to: {save_path}")

2025-11-06 14:42:02,976 - __main__ - INFO - Univariate Analysis: Numerical Distributions
2025-11-06 14:42:07,754 - __main__ - INFO - Univariate analysis completed and saved to: c:\Users\BRAIN\Desktop\dynamic_pricing_project\visualizations\exploration\univariate_analysis_plot.png


### Bivariate Analysis – Understanding Relationships Between Features and the Target Variable

In this step, we perform **Bivariate Analysis** to explore how each feature (both numerical and categorical) relates to the **target variable** — `Historical_Cost_of_Ride`.

Unlike univariate analysis (which studies each variable individually), bivariate analysis focuses on **pairwise relationships** between features and the target.  
This helps us understand how changes in one variable influence the ride cost, guiding feature selection and model design.

#### Goals of this Step:

1. **Examine Correlation (Numerical Features vs Target)**  
   - Compute the correlation matrix to see how strongly each numeric feature is linearly related to the target.  
   - Features with high positive or negative correlation are strong candidates for predictive modeling.

2. **Analyze Category Impact (Categorical Features vs Target)**  
   - Use boxplots to visualize how different categories (e.g., `Location_Category`, `Vehicle_Type`, `Customer_Loyalty_Status`) affect the target.  
   - This helps reveal whether certain groups consistently correspond to higher or lower ride costs.

3. **Simulate Market Dynamics (Demand-Supply Relationship)**  
   - Create a derived feature `Demand_Supply_Ratio_Pre = Number_of_Riders / (Number_of_Drivers + 1)` to model the effect of market conditions.  
   - Visualize this relationship using a scatter plot to assess how supply-demand imbalances impact pricing.

#### Insights We Expect:
- Identify which **numerical variables** most strongly influence ride cost.  
- Detect **category-driven pricing patterns**, such as whether premium vehicle types or loyal customers tend to have higher costs.  
- Understand **demand-supply sensitivity** — a crucial factor in building the dynamic pricing logic.


In [8]:
logger.info("Bivariate Analysis: Relationships with Target Variable")


exploration_dir = os.path.join(VISUALIZATIONS_DIR, "exploration")
os.makedirs(exploration_dir, exist_ok=True)
save_path = os.path.join(exploration_dir, "bivariate_analysis_plot.png")

# Numerical vs Target (Correlation Matrix) 
try:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    if TARGET_COLUMN not in numerical_cols:
        logger.warning(f"Target column '{TARGET_COLUMN}' not numeric — correlation may be invalid.")
    corr_matrix = df[numerical_cols + [TARGET_COLUMN]].corr()

    logger.info("\nCorrelation with Target:\n" + str(
        corr_matrix[TARGET_COLUMN].sort_values(ascending=False).to_markdown()
    ))
except Exception as e:
    logger.error(f"Error computing correlation matrix: {e}")

# Box Plots for Categorical vs Target 
try:
    categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    num_cats = len(categorical_cols)

    if num_cats == 0:
        logger.warning("No categorical columns found for boxplot visualization.")
    else:
        fig, axes = plt.subplots(1, num_cats, figsize=(6 * num_cats, 6))
        if num_cats == 1:
            axes = [axes]

        for i, col in enumerate(categorical_cols):
            sns.boxplot(
                x=col,
                y=TARGET_COLUMN,
                data=df,
                ax=axes[i],
                palette="viridis",
                fliersize=2
            )
            axes[i].set_title(f"{TARGET_COLUMN} by {col}", fontsize=12, fontweight='bold')
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].set_xlabel(col)
            axes[i].set_ylabel(TARGET_COLUMN)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        logger.info(f"Boxplots saved to: {save_path}")
except Exception as e:
    logger.error(f"Error generating boxplots: {e}")

# Demand-Supply Ratio vs Cost (Scatter Plot) 
try:
    if 'Number_of_Riders' in df.columns and 'Number_of_Drivers' in df.columns:
        df['Demand_Supply_Ratio_Pre'] = df['Number_of_Riders'] / (df['Number_of_Drivers'] + 1)

        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            x='Demand_Supply_Ratio_Pre',
            y=TARGET_COLUMN,
            data=df,
            alpha=0.6,
            color="#1f77b4",
            edgecolor=None
        )
        plt.title("Historical Cost vs Pre-Engineered Demand-Supply Ratio", fontsize=13, fontweight='bold')
        plt.xlabel("Demand-Supply Ratio (Pre-feature Engineering)")
        plt.ylabel(TARGET_COLUMN)
        scatter_save_path = os.path.join(exploration_dir, "bivariate_demand_supply_scatter.png")
        plt.savefig(scatter_save_path, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"Scatter plot saved to: {scatter_save_path}")
    else:
        logger.warning("'Number_of_Riders' or 'Number_of_Drivers' column not found.")
except Exception as e:
    logger.error(f"Error plotting demand-supply scatter: {e}")

logger.info("Bivariate analysis completed successfully.")

2025-11-06 14:54:58,328 - __main__ - INFO - Bivariate Analysis: Relationships with Target Variable
2025-11-06 14:54:58,394 - __main__ - ERROR - Error computing correlation matrix: DataFrame.sort_values() missing 1 required positional argument: 'by'
C:\Users\BRAIN\AppData\Local\Temp\ipykernel_6572\2533493426.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\BRAIN\AppData\Local\Temp\ipykernel_6572\2533493426.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\BRAIN\AppData\Local\Temp\ipykernel_6572\2533493426.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=

### Multivariate Analysis – Exploring Interactions Among Multiple Features

In this step, we perform **Multivariate Analysis** to examine how multiple variables interact simultaneously and collectively influence the **target variable** (`Historical_Cost_of_Ride`).

While univariate and bivariate analyses help us understand individual or pairwise relationships, **multivariate analysis** provides a deeper, system-wide view — showing how combinations of features work together to explain variability in ride pricing.

#### Objectives:
1. **Detect Complex Relationships Across Multiple Features**  
   - Understand whether certain variables jointly impact pricing (e.g., `Vehicle_Type` + `Location_Category` or `Number_of_Riders` + `Time_of_Booking`).  
   - Identify interaction effects that may not be visible in single-variable views.

2. **Assess Overall Feature Interdependencies**  
   - Use a correlation **heatmap** to visualize how numerical variables relate to one another.  
   - Detect multicollinearity — variables that provide redundant information — which can negatively affect model stability.

3. **Visualize High-Dimensional Relationships**  
   - Generate **pair plots** and **cluster visualizations** to see how multiple features interact visually.  
   - This helps identify clusters or natural groupings (e.g., high-demand zones, low-rider regions, premium user segments).

#### Insights We Expect:
- Discover **combinations of features** that have a joint influence on ride cost.  
- Identify **redundant or highly correlated features** for potential dimensionality reduction.  
- Gain intuition about **feature clusters** that may form the basis for segmentation or reinforcement learning environments.

#### Output:
- **Correlation Heatmap:** `visualizations/exploration/multivariate_correlation_heatmap.png`  
- **Pairplot of Key Features:** `visualizations/exploration/multivariate_pairplot.png`

These results will guide the **feature selection and model optimization** stages, ensuring that our dynamic pricing model captures both simple and complex feature interactions.


In [9]:
logger.info("Multivariate Analysis: Feature Interactions")

# Select numerical columns for correlation study
numerical_cols = df.select_dtypes(include="number").columns.tolist()
if TARGET_COLUMN not in numerical_cols:
    numerical_cols.append(TARGET_COLUMN)

# Correlation Heatmap 
logger.info("Computing correlation matrix among numerical features...")
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap: Numerical Features", fontsize=14, weight="bold")

# Save heatmap
heatmap_path = os.path.join(VISUALIZATIONS_DIR, "exploration", "multivariate_correlation_heatmap.png")
plt.tight_layout()
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.close()

logger.info(f"Correlation heatmap saved to: {heatmap_path}")

# Pairplot for Numerical Feature Interactions
logger.info("Generating pairplot for key numerical features...")

# Select top 5 correlated features with target for focused pairplot
top_features = corr_matrix[TARGET_COLUMN].abs().sort_values(ascending=False).head(5).index.tolist()

sns.pairplot(df[top_features], diag_kind="kde", corner=True, plot_kws={"alpha": 0.5})
plt.suptitle("Pairplot of Key Numerical Features", y=1.02, fontsize=14, weight="bold")

# Save pairplot
pairplot_path = os.path.join(VISUALIZATIONS_DIR, "exploration", "multivariate_pairplot.png")
plt.savefig(pairplot_path, dpi=300, bbox_inches="tight")
plt.close()

logger.info(f"Pairplot saved to: {pairplot_path}")
logger.info("Multivariate Analysis completed successfully.\n")

2025-11-06 15:16:46,172 - __main__ - INFO - Multivariate Analysis: Feature Interactions
2025-11-06 15:16:46,176 - __main__ - INFO - Computing correlation matrix among numerical features...
2025-11-06 15:16:47,803 - __main__ - INFO - Correlation heatmap saved to: c:\Users\BRAIN\Desktop\dynamic_pricing_project\visualizations\exploration\multivariate_correlation_heatmap.png
2025-11-06 15:16:47,805 - __main__ - INFO - Generating pairplot for key numerical features...
2025-11-06 15:16:52,867 - __main__ - INFO - Pairplot saved to: c:\Users\BRAIN\Desktop\dynamic_pricing_project\visualizations\exploration\multivariate_pairplot.png
2025-11-06 15:16:52,868 - __main__ - INFO - Multivariate Analysis completed successfully.



### Time-Series Analysis

In this step, we perform Time-Series Analysis to uncover temporal trends and seasonality patterns in the ride demand and fare behavior.
Since ride-sharing pricing fluctuates based on when and how frequently rides occur, understanding these patterns is essential for dynamic fare optimization.

This analysis helps us:

Detect daily demand fluctuations — identifying high-traffic days.

Capture hourly and weekly seasonality — understanding when prices typically surge.

Reveal peak demand periods that can inform real-time dynamic pricing models.

Prepare the dataset for forecasting models such as Prophet or LSTM (future steps).

In [11]:
logger.info("Time-Series Analysis: Temporal Patterns and Seasonality")

# Convert time column to categorical for exploration
df['time_of_day'] = df[TIME_COLUMN].astype('category')

# If you have a date column, use it; otherwise create a simple index for daily trends
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    # Total Daily Ride Demand
    logger.info("Plotting total daily demand trends...")
    daily_demand = (
        df.groupby('date')
        .size()
        .reset_index(name='Total_Rides')
        .sort_values('date')
    )

    plt.figure(figsize=(12, 6))
    sns.lineplot(x='date', y='Total_Rides', data=daily_demand, color='steelblue')
    plt.title('Overall Daily Ride Demand Over Time', fontsize=14, weight='bold')
    plt.xlabel('Date')
    plt.ylabel('Number of Rides')
    plt.grid(alpha=0.3)

    daily_demand_path = os.path.join(VISUALIZATIONS_DIR, 'exploration', 'time_series_daily_demand.png')
    plt.tight_layout()
    plt.savefig(daily_demand_path, dpi=300, bbox_inches="tight")
    plt.close()
    logger.info(f"Daily demand trend saved at: {daily_demand_path}")

# Hourly / Time-of-Day Patterns
logger.info("Analyzing time-of-day seasonality patterns...")

hourly_avg = df.groupby('time_of_day')[TARGET_COLUMN].mean().reset_index()
plt.figure(figsize=(8, 5))
sns.barplot(x='time_of_day', y=TARGET_COLUMN, data=hourly_avg, color='darkorange')
plt.title(f'Average {TARGET_COLUMN} by Time of Day', fontsize=13, weight='bold')
plt.xlabel('Time of Day')
plt.ylabel(f'Average {TARGET_COLUMN}')
plt.grid(alpha=0.3)

time_of_day_path = os.path.join(VISUALIZATIONS_DIR, 'exploration', 'time_of_day_avg.png')
plt.tight_layout()
plt.savefig(time_of_day_path, dpi=300, bbox_inches="tight")
plt.close()
logger.info(f"Time-of-day seasonality plot saved at: {time_of_day_path}")

# Weekly Patterns (if you have day info)
if 'day_of_week' in df.columns:
    df['day_name'] = df['day_of_week'].map({0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'})
    weekly_avg = (
        df.groupby('day_name')[TARGET_COLUMN]
        .mean()
        .reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
        .reset_index()
    )

    plt.figure(figsize=(8, 5))
    sns.barplot(x='day_name', y=TARGET_COLUMN, data=weekly_avg, color='teal')
    plt.title(f'Average {TARGET_COLUMN} by Day of Week', fontsize=13, weight='bold')
    plt.xlabel('Day of Week')
    plt.ylabel(f'Average {TARGET_COLUMN}')
    plt.grid(alpha=0.3)

    weekly_path = os.path.join(VISUALIZATIONS_DIR, 'exploration', 'weekly_avg.png')
    plt.tight_layout()
    plt.savefig(weekly_path, dpi=300, bbox_inches="tight")
    plt.close()
    logger.info(f"Weekly seasonality plot saved at: {weekly_path}")

logger.info("Time-Series Analysis completed successfully.\n")
logger.info("EDA Notebook execution completed successfully.")

2025-11-06 15:39:11,981 - __main__ - INFO - Time-Series Analysis: Temporal Patterns and Seasonality
2025-11-06 15:39:11,987 - __main__ - INFO - Analyzing time-of-day seasonality patterns...
C:\Users\BRAIN\AppData\Local\Temp\ipykernel_6572\3667841018.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  hourly_avg = df.groupby('time_of_day')[TARGET_COLUMN].mean().reset_index()
2025-11-06 15:39:12,955 - __main__ - INFO - Time-of-day seasonality plot saved at: c:\Users\BRAIN\Desktop\dynamic_pricing_project\visualizations\exploration\time_of_day_avg.png
2025-11-06 15:39:12,957 - __main__ - INFO - Time-Series Analysis completed successfully.

2025-11-06 15:39:12,958 - __main__ - INFO - EDA Notebook execution completed successfully.
